<a href="https://colab.research.google.com/github/Sajjad-Mahmoudi/Thesis/blob/main/binary_heavyUnet_FlowFromDirectory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import h5py
from google.colab import drive
drive.mount('/content/drive')
from sklearn.model_selection import train_test_split
import keras.backend as K
import tensorflow as tf
import time
!pip install tensorflow-addons
import tensorflow_addons as tfa
from matplotlib import pyplot as plt 

In [ ]:
!pip install wandb
import wandb
from wandb.keras import WandbCallback
wandb.login()

In [ ]:
from keras.models import Model
from keras.layers import Input, Conv2D, Conv2DTranspose, BatchNormalization
from keras.layers import Activation, MaxPool2D, Concatenate

# 2 consecutive convolution operations with batch normalization (NOTE: Batch normalization is not in the original paper.
# Most probably, they did not batch normalization because the paper presenting batch normalization was released
# later than this paper)
def conv_block(input, num_filters, kernel_size=(3, 3), activation='relu', **kwargs):
    x = Conv2D(num_filters, kernel_size=kernel_size, padding="same", **kwargs)(input)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)

    x = Conv2D(num_filters, kernel_size=kernel_size, padding="same", **kwargs)(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)

    return x


# Encoder block = Conv_block followed by MaxPooling
def encoder_block(input, num_filters, **kwargs):
    s = conv_block(input, num_filters, **kwargs)
    p = MaxPool2D((2, 2))(s)
    return s, p


# Decoder block = deconv + concat + conv_block
def decoder_block(input, skip_features, num_filters, kernel_size=(2, 2), **kwargs):
    b = Conv2DTranspose(num_filters, kernel_size=kernel_size, strides=2, padding="same", **kwargs)(input)
    d = Concatenate()([b, skip_features])
    d = conv_block(d, num_filters)
    return d


# UNet
# input_shape = (height, width, num_channel)
def build_unet(input_shape: tuple, num_classes, encoder_filters=None, decoder_filters=None, bridge_filters=1024):
    if decoder_filters is None:
        decoder_filters = [512, 256, 128, 64]
    if encoder_filters is None:
        encoder_filters = [64, 128, 256, 512]

    inputs = Input(input_shape)  # 256 x 256 x 1

    # Encoder Block
    s1, p1 = encoder_block(inputs, encoder_filters[0])  # s1: 256 x 256 x 64, p1: 128 x 128 x 64
    s2, p2 = encoder_block(p1, encoder_filters[1])  # s2: 128 x 128 x 128, p2: 64 x 64 x 128
    s3, p3 = encoder_block(p2, encoder_filters[2])  # s3: 64 x 64 x 256, p3: 32 x 32 x 256
    s4, p4 = encoder_block(p3, encoder_filters[3])  # s4: 32 x 32 x 512, p4: 16 x 16 x 512

    # Bridge Block
    b1 = conv_block(p4, bridge_filters)  # b1: 16 x 16 x 1024

    # Decoder Block
    d1 = decoder_block(b1, s4, decoder_filters[0])
    # Conv2DTrans: 32 x 32 x 512 => concat: 32 x 32 x 1024 => Conv2D: 32 x 32 x 512
    d2 = decoder_block(d1, s3, decoder_filters[1])
    # Conv2DTrans: 64 x 64 x 256 => concat: 64 x 64 x 512 => Conv2D: 64 x 64 x 256
    d3 = decoder_block(d2, s2, decoder_filters[2])
    # Conv2DTrans: 128 x 128 x 128 => concat: 128 x 128 x 256 => Conv2D: 128 x 128 x 128
    d4 = decoder_block(d3, s1, decoder_filters[3])
    # Conv2DTrans: 256 x 256 x 64 => concat: 256 x 256 x 128 => Conv2D: 256 x 256 x 64

    if num_classes == 1:  # Binary
        activation = 'sigmoid'
    else:
        activation = 'softmax'

    # Sigmoid Connection
    outputs = Conv2D(num_classes, (1, 1), padding="same", activation=activation)(d4)  # output: 256 x 256 x 1

    model = Model(inputs, outputs, name="U-Net")
    return model


In [ ]:
def read_hdf5(file):
    with h5py.File(file, 'r') as hf:
        data = np.array(hf.get('data'))
        label = np.array(hf.get('label'))
        return data, label

In [ ]:
data, label = read_hdf5('/content/drive/MyDrive/Thesis/binary/step256/patches_256x256_step256')
x_train, X, y_train, Y = train_test_split(data[:1000], label[:1000], train_size=0.7, random_state=24)

In [ ]:
del data, label
x_valid, x_test, y_valid, y_test = train_test_split(X, Y, test_size=0.65, random_state=24)
del X, Y

In [ ]:
tf.keras.backend.clear_session()
h = w = 256
c = 1
model = build_unet((h, w, c), 1)

In [ ]:
def dice_metric(y_pred, y_true):
    intersection = K.sum(K.sum(K.abs(y_true * y_pred), axis=-1))
    union = K.sum(K.sum(K.abs(y_true) + K.abs(y_pred), axis=-1))
    return 2 * intersection / union

In [ ]:
#run.finish()

In [ ]:
configs = dict(
    batch_size = 2,
    epochs = 10,
    lr = 0.001,
    opt = 'Adam',
    loss = 'sigmoid_focal_crossentropy',
    metrics = 'Dice',
    architecture = 'Heavy_UNet',
    dataset = 'EndoVis15+',
    framework = 'tensorflow'
)

run = wandb.init(project='Binary_Seg', name='step256_FL_Dice', config=configs)
config = wandb.config

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=config.lr), loss=tfa.losses.SigmoidFocalCrossEntropy(), metrics=[dice_metric])

In [ ]:
start_time = time.time()

history = model.fit(x_train, y_train, batch_size=config.batch_size, verbose=1, epochs=config.epochs, validation_data=(x_valid, y_valid), shuffle=False,
                    callbacks=[WandbCallback(training_data=(x_train, y_train), validation_data=(x_valid, y_valid),
                                            labels=["Background", "Tool"], 
                                            predictions=2,
                                            input_type='images', 
                                            output_type='segmentation_mask',
                                            log_evaluation=True,  # table of validation data and model's predictions
                                            log_evaluation_frequency=2, 
                                            class_colors=([1.,0.,0.], [0.,0.,1.]),                                      
                                            # log_gradients=True,
                                            # log_weights=True,
                                            monitor='val_loss'
                                            )])
#model.save('/content/drive/MyDrive/Thesis/binary/step256/epoch100_step256_sigmoidFocalCE_heavyUNet.hdf5')

seconds = time.time() - start_time
print('Time Taken:', time.strftime("%H:%M:%S",time.gmtime(seconds)))

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
wandb.log({'test_loss': test_loss, 'test_accuracy': test_acc})
run.join()

In [ ]:
run.finish()

In [ ]:
np.save('/content/drive/MyDrive/Thesis/binary/step256/HISTORY(epoch100_step256_sigmoidFocalCE_heavyUNet).npy', history.history)
#history=np.load('/content/drive/MyDrive/Thesis/binary/step256/HISTORY(epoch100_step256_sigmoidFocalCE_heavyUNet).npy',allow_pickle='TRUE').item()

In [ ]:
'''# plot the training and validation accuracy and loss at each epoch
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'y', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

acc = history.history['dice_metric']
#acc = history.history['accuracy']
val_acc = history.history['val_dice_metric']
#val_acc = history.history['val_accuracy']

plt.plot(epochs, acc, 'y', label='Training Dice')
plt.plot(epochs, val_acc, 'r', label='Validation Dice')
plt.title('Training and validation Dice')
plt.xlabel('Epochs')
plt.ylabel('Dice')
plt.legend()
plt.show()'''